In [ ]:
# 원본 JSON → npy 변환
# GCS의 raw/keypoints/batch_01/ 에 저장된 AI-Hub 원본 키포인트 JSON 파일들을 읽어서 numpy 배열로 변환

In [ ]:
# Step 1. GCS 연결, 인증 후 버킷 연결
from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
import json

PROJECT_ID = "project-id"
BUCKET_NAME = "sign-language-data-2026"

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

print("GCS 연결")

In [ ]:
# Step 2. GCS에서 label_map 불러오기 
blob = bucket.blob("processed/label_map.json")
label_map = json.loads(blob.download_as_text())
print(f"label_map 불러오기 완료: {len(label_map)}개 단어")
print(f"예시: {dict(list(label_map.items())[:5])}")

In [ ]:
# keypoint 파일 샘플 확인 (JSON 구조)
import json

blobs = list(bucket.list_blobs(prefix="raw/keypoints/batch_01/", max_results=1))
sample = blobs[0]
print("파일명:", sample.name)

content = json.loads(sample.download_as_text())
print("최상위 키:", list(content.keys()))
print("샘플 내용:", json.dumps(content, ensure_ascii=False, indent=2)[:1000])

In [ ]:
# people 안에 어떤 키가 있는지 확인
blobs = list(bucket.list_blobs(prefix="raw/keypoints/batch_01/", max_results=1))
content = json.loads(blobs[0].download_as_text())

people = content["people"]
print("people 타입:", type(people))
print("people 키 목록:", list(people.keys()))

# 각 키포인트 배열 길이 확인
for key in people.keys():
    if isinstance(people[key], list):
        print(f"  {key}: {len(people[key])}개 값 → {len(people[key])//3}개 포인트 (x,y,conf)")

In [ ]:
# WORD0001 폴더에 프레임이 몇 개인지 확인
blobs = list(bucket.list_blobs(prefix="raw/keypoints/batch_01/NIA_SL_WORD0001_REAL01_D/"))
print(f"WORD0001 프레임 수: {len(blobs)}개")
print("첫 파일:", blobs[0].name.split("/")[-1])
print("끝 파일:", blobs[-1].name.split("/")[-1])

# 폴더(영상) 종류 확인 - WORD0001이 몇 개 영상으로 구성되는지
import re
folders = set()
for b in bucket.list_blobs(prefix="raw/keypoints/batch_01/NIA_SL_WORD0001_"):
    folder = b.name.split("/")[3]
    folders.add(folder)
print(f"\nWORD0001 영상 폴더들: {sorted(folders)}")

In [ ]:
# 각 각도별 프레임 수 + WORD 번호가 몇 개나 있는지 확인
angles = ['D', 'F', 'L', 'R', 'U']
for angle in angles:
    blobs = list(bucket.list_blobs(prefix=f"raw/keypoints/batch_01/NIA_SL_WORD0001_REAL01_{angle}/"))
    print(f"WORD0001_{angle}: {len(blobs)}프레임")

# 전체 폴더 개수 (= 총 영상 수)
all_folders = set()
for b in bucket.list_blobs(prefix="raw/keypoints/batch_01/", delimiter="/"):
    pass
# 다른 방법으로
import re
sample_blobs = list(bucket.list_blobs(prefix="raw/keypoints/batch_01/NIA_SL_WORD0002_"))
folders2 = set()
for b in sample_blobs:
    folders2.add(b.name.split("/")[3])
print(f"\nWORD0002 폴더들: {sorted(folders2)}")

In [ ]:
# Step3 함수 정의 + 오류 탐지
import numpy as np
from collections import Counter

FEATURE_SIZE = 274
TARGET_FRAMES = 30
MIN_FRAMES = 10

errors = []

def extract_keypoints(people, filepath=""): # 각 키포인트 배열에서 신뢰도 제거
    """신뢰도 제거하고 x, y만 추출 (274개)"""
    def remove_confidence(keypoints):
        return [keypoints[i] for i in range(len(keypoints)) if i % 3 != 2]

    left_hand  = remove_confidence(people.get("hand_left_keypoints_2d", []))
    right_hand = remove_confidence(people.get("hand_right_keypoints_2d", []))
    face       = remove_confidence(people.get("face_keypoints_2d", []))
    pose       = remove_confidence(people.get("pose_keypoints_2d", []))

    result = left_hand + right_hand + face + pose

    # ── 오류 탐지 ──────────────────────────────
    # 1. 개수 확인
    if len(result) != FEATURE_SIZE:
        errors.append({
            "type": "키포인트 개수 오류",
            "file": filepath,
            "error": f"{len(result)}개 (274개여야 함)"
        })
        return None

    arr = np.array(result)

    # 2. NaN 확인
    if np.any(np.isnan(arr)):
        errors.append({
            "type": "NaN 값 포함",
            "file": filepath,
            "error": "숫자가 아닌 값 포함"
        })
        return None

    # 3. Inf 확인
    if np.any(np.isinf(arr)):
        errors.append({
            "type": "Inf 값 포함",
            "file": filepath,
            "error": "무한대 값 포함"
        })
        return None

    # 4. 전부 0인지 확인
    if np.all(arr == 0):
        errors.append({
            "type": "빈 프레임",
            "file": filepath,
            "error": "키포인트 전부 0"
        })
        return None

    return result


def normalize_frames(frames): # 프레임 30으로 통일
    """프레임 수를 30으로 정규화"""
    frames = np.array(frames)
    n = len(frames)

    if n == 0:
        return np.zeros((TARGET_FRAMES, FEATURE_SIZE))
    elif n >= TARGET_FRAMES: # 30프레임 초과하면 균등 샘플링으로 30개 추출
        indices = np.linspace(0, n-1, TARGET_FRAMES, dtype=int)
        return frames[indices]
    else: # 30 프레임 미만이면 뒤를 0으로 제로패딩
        padded = np.zeros((TARGET_FRAMES, FEATURE_SIZE))
        padded[:n] = frames
        return padded


def validate_word_samples(word_id, samples, angle_results): # 단어별로 처리 성공/실패 여부, label_map 누락 여부 검증
    """단어별 샘플 검증"""
    # 각도별 성공/실패 출력
    for angle, success in angle_results.items():
        if not success:
            errors.append({
                "type": "각도 처리 실패",
                "file": f"NIA_SL_WORD{word_id:04d}_REAL01_{angle}/",
                "error": f"유효한 프레임 부족"
            })

    # label_map에 있는 단어인지 확인
    if str(word_id) not in label_map:
        errors.append({
            "type": "label_map 누락",
            "file": f"WORD{word_id:04d}",
            "error": "label_map에 없는 단어"
        })

    return len(samples) > 0


print("함수 정의 완료")
print(f"  - extract_keypoints()     : 키포인트 추출 + 오류 탐지 (파일명 포함)")
print(f"  - normalize_frames()      : 프레임 정규화 (→ {TARGET_FRAMES}프레임)")
print(f"  - validate_word_samples() : 단어별 샘플 검증")

In [ ]:
# GCS에서 로컬로 다운로드

import subprocess
import sys
import os

# 폴더 먼저 생성
os.makedirs("/content/keypoints", exist_ok=True)
print("폴더 생성 완료")

print("다운로드 시작... (약 30분 소요)")
print("-" * 50)

process = subprocess.Popen([
    "gsutil", "-m", "cp", "-r", #Colab 로컬로 다운로드
    "gs://sign-language-data-2026/raw/keypoints/batch_01/",
    "/content/keypoints/"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    line = line.strip()
    if line:
        print(line)
        sys.stdout.flush()

process.wait()
print("-" * 50)
print("다운로드 완료")

result = subprocess.getoutput("du -sh /content/keypoints/")
print(f"다운로드 크기: {result}")

In [ ]:
import subprocess
print(subprocess.getoutput("du -sh /content/*"))

202M	/content/keypoints
55M	/content/sample_data


In [ ]:
import subprocess
print(subprocess.getoutput("df -h /"))
print("---")
print(subprocess.getoutput("du -sh /* 2>/dev/null"))

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G  108G     0 100% /
---
0	/bin
4.0K	/boot
256M	/content
51M	/datalab
0	/dev
87G	/etc
4.0K	/home
8.0K	/kaggle
0	/lib
0	/lib32
0	/lib64
0	/libx32
4.0K	/media
4.0K	/mnt
109M	/opt
0	/proc
21M	/python-apt
340K	/python-apt.tar.xz
3.7G	/root
84K	/run
0	/sbin
4.0K	/srv
0	/sys
40K	/tmp
1.8G	/tools
16G	/usr
131M	/var


In [ ]:
import subprocess
print(subprocess.getoutput("du -sh /etc/* 2>/dev/null | sort -rh | head -20"))

87G	/etc/dev
816K	/etc/ssl
500K	/etc/java-17-openjdk
200K	/etc/fonts
140K	/etc/apt
108K	/etc/X11
84K	/etc/pam.d
72K	/etc/mime.types
68K	/etc/systemd
68K	/etc/requirements3.txt
64K	/etc/libibverbs.d
56K	/etc/ld.so.cache
56K	/etc/iproute2
52K	/etc/security
40K	/etc/polkit-1
40K	/etc/dpkg
36K	/etc/sysctl.d
36K	/etc/R
36K	/etc/alternatives
32K	/etc/init.d


In [ ]:
import subprocess
print(subprocess.getoutput("du -sh /etc/dev/* 2>/dev/null | sort -rh | head -20"))

87G	/etc/dev/urandom
0	/etc/dev/zero
0	/etc/dev/stdout
0	/etc/dev/stdin
0	/etc/dev/stderr
0	/etc/dev/random
0	/etc/dev/ptmx
0	/etc/dev/null
0	/etc/dev/fuse
0	/etc/dev/full
0	/etc/dev/fd
0	/etc/dev/core


In [ ]:
import subprocess
print(subprocess.getoutput("df -h /"))
print("---")
print(subprocess.getoutput("du -sh /content/*"))


Filesystem      Size  Used Avail Use% Mounted on
overlay         108G  108G     0 100% /
---
202M	/content/keypoints
55M	/content/sample_data
